# 02 — Feature Engineering

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shopska/Personalized_anaysis_of_social_media/blob/main/notebooks/02-feature-engineering.ipynb)

**Мета:** перетворити таблицю з EDA на таблицю ознак для моделей. Головне правило — кожна ознака має бути **відома в момент публікації відео** (інакше це data leakage).

**Вхід:** `data/interim/videos_with_target.parquet` (результат `01-eda`) + таблиця `creator_daily`.

**Вихід:** `data/processed/features.parquet` — id + 2 цілі (`log_views`, `engagement_rate`) + ознаки.

**Висновки:** *(заповнити після запуску — див. кінець ноутбука)*

> ⚠️ Markdown українською — тимчасовий, перед здачею перекладемо англійською, як у `01-eda`.

## 0. Setup

Те саме, що в `01-eda`, але блок обгорнутий в `if IN_COLAB`, щоб ноутбук запускався і локально (так вимагає `notebooks/README.md`).

**Нюанси Colab:**
- Після першого `pip install` → *Runtime → Restart session* → запустити все ще раз (інакше підтягнуться старі numpy/pandas).
- Увесь код ознак лежить у `src/creator_analytics/features.py`. `%autoreload` у Colab не працює, тому **якщо ти змінила щось у `src/` — треба `git pull` + Restart session**, інакше Python використовує стару версію модуля.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
REPO = "Personalized_anaysis_of_social_media"

if IN_COLAB:
    from google.colab import userdata

    REPO_DIR = f"/content/{REPO}"
    if not os.path.exists(REPO_DIR):
        token = userdata.get("GITHUB_TOKEN")
        !git clone -q https://{token}@github.com/Shopska/{REPO}.git {REPO_DIR}
    %cd {REPO_DIR}
    !git pull -q
    !pip install -q -r requirements.txt
    !pip install -q -e .
    sys.path.insert(0, f"{REPO_DIR}/src")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # run from the repo root so that relative paths in the config work

In [ ]:
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from creator_analytics import features as F
from creator_analytics.config import load_config
from creator_analytics.data import filter_creators, load_cached_table
from creator_analytics.utils import set_seed

cfg = load_config("config/default.yaml")
set_seed(cfg["seed"])
AUTHOR = cfg["data"]["author_column"]
FCFG = cfg["features"]

pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
print(FCFG)

## 1. Завантаження даних

`videos_with_target.parquet` ми зберегли на Drive в кінці `01-eda`, бо `/content` стирається після кожної сесії. Звідти ж беремо `creator_daily` (функція `load_cached_table` з `data.py`: спершу шукає локально, потім на Drive, і лише тоді качає з HuggingFace).

Таблицю `engagement_daily` тут **не завантажуємо**: ціль уже порахована в EDA, а щоденні перегляди як ознаки — це leakage.

In [ ]:
DATA_CACHE = None
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    DATA_CACHE = Path("/content/drive/MyDrive/creator_analytics/data")

INTERIM = Path(cfg["paths"]["interim_data"])
if not INTERIM.exists():
    cached = DATA_CACHE / INTERIM.name if DATA_CACHE else None
    if cached is None or not cached.exists():
        raise FileNotFoundError(f"{INTERIM} not found — run notebooks/01-eda.ipynb first")
    INTERIM.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(cached, INTERIM)

videos = pd.read_parquet(INTERIM)
creators = load_cached_table(
    "creator_daily", "data/raw", cfg["data"]["hf_repo"], cache_dir=DATA_CACHE
)
print(f"videos:   {videos.shape}")
print(f"creators: {creators.shape}")

## 2. Що НЕ можна давати моделі (leakage)

Прибираємо все, що вимірюється **після** публікації: лічильники переглядів/лайків/коментарів, їхні rate-и і `observed_day`.

Окремо: `follower_count` з EDA ми брали з *найближчої* дати, яка може бути **після** публікації. Тут викидаємо його і рахуємо заново, лише з минулих дат (розділ 6). Старий стовпчик зберігаємо в `eda_followers` лише для порівняння.

Текстові стовпчики (`transcript`, `gpt_summary`, `desc`) у модель напряму не йдуть — з них беремо прості числові ознаки (довжина, наявність). Ембедінги тексту — можливий nice-to-have пізніше.

In [ ]:
eda_followers = videos["follower_count"].copy()
leaky_present = [c for c in F.LEAKY_COLUMNS if c in videos.columns]
df = videos.drop(columns=leaky_present)
print("Dropped:", leaky_present)

## 3. Час публікації

Годину кодуємо як точку на колі (`hour_sin`, `hour_cos`), щоб модель «знала», що 23:00 і 00:00 — сусіди, а не різниця в 23 одиниці. Плюс день тижня і прапорець вихідного.

В EDA бачили: вдень (15–18) краще, ніж уночі; день тижня майже не впливає.

In [ ]:
df = F.add_time_features(df)
display(df[F.TIME_FEATURES].describe().round(3))

## 4. Контент і текст

- `resolution` — з `ratio` ("540p" → 540).
- `desc_length`, `has_desc`, `n_hashtags` — з опису. `n_hashtags` рахуємо зі стовпця `hashtags`, бо він **повний**, а `hashtag_count` пропущений у 38% відео.
- `has_sticker_text`, `sticker_text_length`, `has_transcript` — «нема тексту/мови» це інформація, а не помилка (висновок EDA).
- 5 текстових статистик пропущені **разом** у 38% відео → залишаємо `NaN` (RandomForest і XGBoost вміють з ним працювати) + прапорець `has_text_stats`. Заповнити нулем було б неправильно: 0 слів ≠ «невідомо».
- Емоції — як є + прапорець `has_emotions` (пропущені лише в 0.5%).

Перевірка нижче: `n_hashtags` має сильно корелювати з `hashtag_count` там, де обидва є — це підтверджує, що ми рахуємо те саме.

In [ ]:
df = F.add_content_features(df)
display(df[F.CONTENT_FEATURES].describe().T.round(3))

both = df[["n_hashtags", "hashtag_count"]].dropna()
print("Spearman(n_hashtags, hashtag_count):", round(both.corr(method="spearman").iloc[0, 1], 3))

## 5. Музика

- `is_own_sound` — креатор використовує **власний** звук (`music_owner_id == author_id`) чи чужий.
- `music_uses_prev_window` — скільки відео в датасеті використали цей самий звук за попередні 7 днів, тобто наскільки звук «у тренді» на момент публікації. Рахуємо лише **минуле**. Обмеження: у датасеті вибірка креаторів, а не весь TikTok, тож це грубий проксі.

`music_selected_from` (проксі шаблону монтажу) закодуємо one-hot у розділі 8.

In [ ]:
df = F.add_music_features(df, trend_window_days=FCFG["music_trend_window_days"])

print(f"Own sound: {df['is_own_sound'].mean():.1%} of videos")
display(df.groupby("is_own_sound")["log_views"].agg(["size", "median"]).round(2))
display(df["music_uses_prev_window"].describe(percentiles=[.5, .9, .99]).round(1).to_frame())

## 6. Розмір акаунта — лише з минулого

Беремо **останній знімок `creator_daily` строго до дня публікації** (`merge_asof`, `direction="backward"`, без того самого дня — бо знімок дня публікації міг бути зроблений вже після виходу відео).

Ознаки: `log_followers`, `log_following`, `log_total_favorited`, `account_video_count`. Логарифм — щоб SHAP-графіки були читабельні (від 1k до 30M фоловерів).

Перевіряємо: (1) для якої частки відео є попередній знімок; (2) наскільки нова версія збігається з EDA-версією (має бути дуже близько — фоловери змінюються повільно).

**Якщо частка без знімка велика (>10%)** — напиши, обговоримо fallback (наприклад, найближчий знімок + прапорець).

In [ ]:
df = F.add_follower_features(df, creators, author_column=AUTHOR)

print(f"Videos with an earlier account snapshot: {df['has_account_stats'].mean():.1%}")
pair = pd.DataFrame({"new": df["log_followers"], "eda": np.log1p(eda_followers)}).dropna()
print("Spearman(new, EDA nearest-date followers):", round(pair.corr(method="spearman").iloc[0, 1], 4))
print("Spearman(log_followers, log_views):", round(df["log_followers"].corr(df["log_views"], method="spearman"), 3))

## 7. Історія креатора — лише попередні відео

EDA показав: **креатор пояснює 74% дисперсії** переглядів. Тому глобальній моделі потрібні ознаки «який цей креатор зазвичай», інакше порівняння з per-creator моделями буде нечесним.

Для кожного відео рахуємо по відео **того самого креатора, опублікованих щонайменше 7 днів раніше**:
- `hist_n_videos` — скільки таких відео;
- `hist_mean_log_views`, `hist_std_log_views` — типовий рівень і розкид;
- `hist_recent_mean_log_views` — середнє останніх 5 («поточна форма»);
- `hist_mean_engagement_rate`.

**Чому 7 днів, а не «всі попередні»:** перегляди ростуть ~2 тижні (EDA: ~86% на 5-й день, ~92% на 10-й). Фінальні перегляди вчорашнього відео ще не існували, коли публікувалося сьогоднішнє. Розрив задається в `config/default.yaml` (`history_gap_days`).

Ритм публікацій (без цілі, тому без розриву): `days_since_prev_video`, `n_videos_prev_7d`.

Ознаки рахуємо на **всіх** 209k відео, до фільтра ≥30: на історію конкретного креатора інші креатори не впливають, а тренд звуку (розділ 5) якраз потребує всіх.

Перевірка коректності нижче: для випадкового відео перераховуємо середнє «вручну» і порівнюємо. Плюс у репозиторії є тести `tests/test_features.py`, які перевіряють, що ознаки не залежать від цілі самого відео і від майбутніх відео.

In [ ]:
df = F.add_creator_history_features(
    df,
    author_column=AUTHOR,
    gap_days=FCFG["history_gap_days"],
    recent_k=FCFG["history_recent_k"],
)
display(df[F.HISTORY_FEATURES].describe().T.round(3))
print("Share of videos without history (first week of each creator):",
      f"{df['hist_mean_log_views'].isna().mean():.1%}")

In [ ]:
# Manual check on a few random videos
times = pd.to_datetime(df["create_time"])
gap = pd.Timedelta(days=FCFG["history_gap_days"])
for i in df.sample(5, random_state=cfg["seed"]).index:
    mask = (df[AUTHOR] == df.at[i, AUTHOR]) & (times < times[i] - gap)
    manual = df.loc[mask, "log_views"].mean()
    print(f"video {i}: feature={df.at[i, 'hist_mean_log_views']:.4f}  manual={manual:.4f}")

In [ ]:
d = df.dropna(subset=["hist_mean_log_views"])
fig, ax = plt.subplots(figsize=(7, 5))
hb = ax.hexbin(d["hist_mean_log_views"], d["log_views"], gridsize=50, bins="log", cmap="viridis")
ax.plot([0, 16], [0, 16], color="red", lw=1, label="y = x")
ax.set_xlabel("hist_mean_log_views (creator's earlier videos)")
ax.set_ylabel("log(1 + views)")
ax.set_title("Creator's past level vs this video")
ax.grid(False)
ax.legend()
fig.colorbar(hb, label="log10(count)")
plt.show()
print("Spearman:", round(d["hist_mean_log_views"].corr(d["log_views"], method="spearman"), 3))

## 8. Категоріальні ознаки → one-hot

`topic` і `music_selected_from` кодуємо one-hot: RandomForest у sklearn не приймає текст, а для SHAP кожна категорія отримує власне значення («музика з edit_page_recent: +0.3» — легко пояснити). Рідкісні категорії (< 500 відео) зливаємо в `other`, щоб не захаращувати модель і графіки.

Тут же збираємо фінальну таблицю функцією `build_features` — вона робить **рівно ті самі кроки**, що й розділи 2–8. Перевіряємо, що результат збігається з покроковим: так гарантуємо, що наступні ноутбуки й CLI отримають ідентичні ознаки.

In [ ]:
table = F.build_features(videos, creators, cfg)
FEATURES = F.get_feature_columns(table)

step_cols = [c for c in FEATURES if c in df.columns]
pd.testing.assert_frame_equal(
    table[step_cols], df[step_cols].reset_index(drop=True), check_dtype=False
)
print("build_features == step-by-step result ✔")

groups = {
    "content": F.CONTENT_FEATURES,
    "time": F.TIME_FEATURES,
    "music": F.MUSIC_FEATURES,
    "account": F.ACCOUNT_FEATURES,
    "history": F.HISTORY_FEATURES,
    "one-hot": [c for c in FEATURES if "__" in c],
}
display(pd.Series({k: len(v) for k, v in groups.items()}, name="features").to_frame())
print(f"Total: {len(FEATURES)} features, {len(table):,} videos")
print(groups["one-hot"])

## 9. Перевірка якості ознак

1. **Пропуски** — у яких ознаках і скільки (очікувано: текстові статистики ~38%, історія — перший тиждень кожного креатора).
2. **Константні ознаки** — не несуть інформації, мають бути відсутні.
3. **Дублюючі ознаки** (|Spearman| > 0.9). Для точності дерев це не страшно, але **SHAP ділить «заслугу» між корельованими ознаками** — це треба пам'ятати при інтерпретації. З EDA вже знаємо пару `speaking_rate` ↔ `has_transcript`. Залишаємо обидві, але при інтерпретації SHAP розглядаємо їх разом.

In [ ]:
missing = table[FEATURES].isna().mean().sort_values(ascending=False)
missing = missing[missing > 0]
ax = missing.plot.barh(figsize=(8, 0.35 * len(missing) + 1))
ax.invert_yaxis()
ax.set_xlabel("share missing")
ax.set_title("Missing values in features")
plt.show()

constant = [c for c in FEATURES if table[c].nunique(dropna=True) <= 1]
print("Constant features:", constant or "none")

In [ ]:
sample = table[FEATURES].sample(min(50_000, len(table)), random_state=cfg["seed"])
corr = sample.corr(method="spearman").abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
pairs = upper.stack().loc[lambda s: s > 0.9].sort_values(ascending=False)
display(pairs.round(3).to_frame("|spearman|") if len(pairs) else "No pairs above 0.9")

## 10. Чи несуть нові ознаки сигнал? Глобально vs всередині креатора

Повторюємо ідею з розділу 9 EDA для **нових** ознак: кореляція з `log_views` по всьому датасету vs медіана кореляцій всередині кожного креатора (≥30 відео).

Очікування:
- ознаки акаунта й історії (`log_followers`, `hist_mean_log_views`) — дуже сильні **глобально**, але всередині креатора майже константні → корисні для глобальної моделі, марні для per-creator;
- `hist_recent_mean_log_views`, `days_since_prev_video`, `music_uses_prev_window` можуть мати сигнал і **всередині** креатора.

Це підказка для моделювання, а не доказ причинності. Рахуємо швидким векторним способом (ранги всередині креатора → Pearson = Spearman).

Нюанс для `hist_mean_log_views` всередині креатора: навіть на чистому шумі медіана буде трохи **від'ємною**. Це статистичний артефакт, а не leakage: середнє креатора фіксоване, тож якщо поточне відео вище середнього, решта (зокрема його «історія») в середньому нижча.

In [ ]:
import warnings

warnings.filterwarnings("ignore", message="An input array is constant")  # creators with a constant feature

dfc = filter_creators(table, AUTHOR, cfg["data"]["min_videos_per_creator"])
NEW = (["desc_length", "n_hashtags", "sticker_text_length", "resolution"]
       + F.TIME_FEATURES + F.MUSIC_FEATURES + F.ACCOUNT_FEATURES[:4] + F.HISTORY_FEATURES)


def within_creator_spearman(d, feat, target="log_views"):
    d = d[[AUTHOR, feat, target]].dropna()
    g = d.groupby(AUTHOR)
    x = g[feat].rank() - g[feat].rank().groupby(d[AUTHOR]).transform("mean")
    y = g[target].rank() - g[target].rank().groupby(d[AUTHOR]).transform("mean")
    num = (x * y).groupby(d[AUTHOR]).sum()
    den = np.sqrt((x**2).groupby(d[AUTHOR]).sum() * (y**2).groupby(d[AUTHOR]).sum())
    return (num / den.replace(0, np.nan)).dropna()


rows = []
for f in NEW:
    per_creator = within_creator_spearman(dfc, f)
    rows.append({
        "feature": f,
        "global": dfc[f].corr(dfc["log_views"], method="spearman"),
        "median per-creator": per_creator.median(),
        "share of creators where defined": len(per_creator) / dfc[AUTHOR].nunique(),
    })
signal = pd.DataFrame(rows).set_index("feature").round(3)
display(signal.sort_values("global", key=abs, ascending=False))

## 11. Збереження

Зберігаємо в `data/processed/features.parquet` (так описано в `data/README.md`) і копію на Drive. Дані в git не комітимо — це вже налаштовано в `.gitignore`.

**Перед комітом ноутбука:** *Runtime → Restart session and run all*, переконатися, що все пройшло без помилок (правило з `notebooks/README.md`), і лише тоді зберігати на GitHub — з outputs.

In [ ]:
OUT = Path(cfg["paths"]["features"])
OUT.parent.mkdir(parents=True, exist_ok=True)
table.to_parquet(OUT, index=False)
if DATA_CACHE:
    shutil.copy(OUT, DATA_CACHE / OUT.name)
print(OUT, table.shape)

## Висновки

*(Заповнити після запуску, з реальними числами з виходів вище. Що варто написати:)*

1. **Скільки ознак і яких груп** (таблиця з розділу 8).
2. **Leakage:** які стовпці прибрали і чому; фоловери тепер лише з минулого (частка покриття — розділ 6); історія креатора з розривом 7 днів (перевірено вручну + тестами).
3. **Пропуски:** які ознаки мають NaN і чому це нормально для дерев.
4. **Дублюючі пари** з розділу 9 — як будемо з ними поводитися в SHAP.
5. **Сигнал (розділ 10):** які ознаки сильні глобально, але не всередині креатора, і навпаки — це прямо підтримує ідею порівняння global vs per-creator.

**Наступні кроки (моделювання, `03-...`):**
- Baseline 1: глобальна модель **без** ознак креатора (`CONTENT + TIME + MUSIC + one-hot`).
- Baseline 2: глобальна модель **з** `ACCOUNT + HISTORY` — чесний конкурент для персоналізації.
- Per-creator моделі на тих самих тестових відео → порівняння RMSE / MAE / R².
- Відкрите питання для обговорення: залишити поточний random split, стратифікований по креатору, чи перейти на **часовий** (останні 20% відео кожного креатора — тест). Часовий реалістичніший («прогноз наступного відео») і краще поєднується з історичними ознаками.